Anggota Kelompok:

1. Orchidea Savira Putri Arief Dhidin (25031554174)

2. Dentar Muhammad Ababilla (25031554277)


# *Exercise 4 Text Pre-processing*
 Try to preprocess the text you scrape from the exercise before.

Data yang dipreprocessing berasal dari hasil **Exercise 3**:
1. **HTTP Scraping**  `quotes.toscrape.com` (kolom teks: `quote`)
2. **API Scraping**  `itunes.apple.com ` ( `trackName` (judul lagu) & `artistName` (nama artis)`)




## **1. Persiapan Environment & Import Library**

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import string
import matplotlib.pyplot as plt
from wordcloud import WordCloud

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

print("Semua library berhasil dimuat!")

Semua library berhasil dimuat!


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## **2. Mengambil Data HTTP Scraping (quotes.toscrape.com)**


In [ ]:
base_url = "http://quotes.toscrape.com/page/{}/"

quotes_list = []
authors_list = []
tags_list = []

total_pages = 5

for page in range(1, total_pages + 1):
    target_url = base_url.format(page)
    print(f"[HTTP Scraping] Mengambil data: {target_url}")

    response = requests.get(target_url)
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, "html.parser")
        quote_boxes = soup.find_all("div", class_="quote")

        for box in quote_boxes:
            quote_text = box.find("span", class_="text").get_text(strip=True)
            author_name = box.find("small", class_="author").get_text(strip=True)
            tag_elements = box.find_all("a", class_="tag")
            tags = ", ".join([t.get_text(strip=True) for t in tag_elements])

            quotes_list.append(quote_text)
            authors_list.append(author_name)
            tags_list.append(tags)
    else:
        print(f"Gagal memuat halaman {page} (Status: {response.status_code})")

    time.sleep(0.3)

df_http = pd.DataFrame({
    "quote": quotes_list,
    "author": authors_list,
    "tags": tags_list
})

print(f"\nTotal data scraping HTTP: {len(df_http)} baris")
display(df_http.head())


[HTTP Scraping] Mengambil data: http://quotes.toscrape.com/page/1/
[HTTP Scraping] Mengambil data: http://quotes.toscrape.com/page/2/
[HTTP Scraping] Mengambil data: http://quotes.toscrape.com/page/3/
[HTTP Scraping] Mengambil data: http://quotes.toscrape.com/page/4/
[HTTP Scraping] Mengambil data: http://quotes.toscrape.com/page/5/

Total data scraping HTTP: 50 baris


,quote,author,tags
0,“The world as we have created it is a process ...,Albert Einstein,"change, deep-thoughts, thinking, world"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"abilities, choices"
2,“There are only two ways to live your life. On...,Albert Einstein,"inspirational, life, live, miracle, miracles"
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"aliteracy, books, classic, humor"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"be-yourself, inspirational"


## **3. Mengambil Data API Scraping (music-to-scrape.org)**


In [ ]:
itunes_api_url = "https://itunes.apple.com/search"

search_terms = ["love", "dream", "night", "sky", "fire"]

music_records = []

print("[iTunes API] Mengambil data lagu berdasarkan beberapa kata kunci...")
for term in search_terms:
    params = {
        "term": term,
        "media": "music",
        "entity": "song",
        "limit": 3
    }
    try:
        res = requests.get(itunes_api_url, params=params, timeout=10)
        if res.status_code == 200:
            data = res.json()
            results = data.get("results", [])
            music_records.extend(results)
            print(f"  '{term}': {len(results)} track ditemukan")
        else:
            print(f"  '{term}': gagal (Status: {res.status_code})")
    except requests.exceptions.RequestException as e:
        print(f"  '{term}': error - {e}")
    time.sleep(0.3)

df_music = pd.DataFrame(music_records)
print(f"\nTotal data berhasil diambil via iTunes API: {len(df_music)} baris")

if df_music.empty:
    print("\n[PERINGATAN] df_music KOSONG — iTunes API kemungkinan sedang bermasalah")
    print("(jarang terjadi karena API publik resmi Apple). Menggunakan data cadangan")
    print("(fallback) supaya notebook tetap bisa lanjut ke tahap preprocessing.\n")

    df_music = pd.DataFrame({
        "trackName": [
            "Bohemian Rhapsody", "Imagine", "Hotel California",
            "Billie Jean", "Shape of You"
        ],
        "artistName": [
            "Queen", "John Lennon", "Eagles",
            "Michael Jackson", "Ed Sheeran"
        ]
    })
else:
    display(df_music[["trackName", "artistName", "collectionName"]].head())

print("\nKolom yang tersedia:", list(df_music.columns))


[iTunes API] Mengambil data lagu berdasarkan beberapa kata kunci...
  'love': 3 track ditemukan
  'dream': 3 track ditemukan
  'night': 3 track ditemukan
  'sky': 3 track ditemukan
  'fire': 3 track ditemukan

Total data berhasil diambil via iTunes API: 15 baris


,trackName,artistName,collectionName
0,Love,Keyshia Cole,The Way It Is
1,LOVE. (feat. Zacari),Kendrick Lamar,DAMN.
2,Love,Musiq Soulchild,Aijuswanaseing (Special Edition)
3,(dream),salvia palth,melanchole
4,Dream,"Robxrt, the most romantic man in hanoi & HVGGi",VIRGO9



Kolom yang tersedia: ['wrapperType', 'kind', 'artistId', 'collectionId', 'trackId', 'artistName', 'collectionName', 'trackName', 'collectionCensoredName', 'trackCensoredName', 'artistViewUrl', 'collectionViewUrl', 'trackViewUrl', 'previewUrl', 'artworkUrl30', 'artworkUrl60', 'artworkUrl100', 'collectionPrice', 'trackPrice', 'releaseDate', 'collectionExplicitness', 'trackExplicitness', 'discCount', 'discNumber', 'trackCount', 'trackNumber', 'trackTimeMillis', 'country', 'currency', 'primaryGenreName', 'isStreamable', 'collectionArtistName', 'collectionArtistId', 'collectionArtistViewUrl']


## **4. Text Pre-processing**
- **Case Folding** (mengubah semua huruf menjadi huruf kecil/lowercase)
- **Cleaning** (menghapus tanda baca, angka, dan karakter khusus)
- **Tokenization** (memecah teks menjadi token/kata)
- **Stopword Removal** (menghapus kata-kata umum yang tidak bermakna)
- **Lemmatization** (mengembalikan kata ke bentuk dasarnya)


In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

df_http['quote_raw'] = df_http['quote']

text_col_candidates = [c for c in ['trackName', 'artistName', 'name', 'artists'] if c in df_music.columns]
print("Kolom teks yang dipakai untuk preprocessing music:", text_col_candidates)

def combine_text_columns(row):
    parts = []
    for col in text_col_candidates:
        val = row[col]
        if isinstance(val, list):
            val = " ".join(str(v) for v in val)
        parts.append(str(val))
    return " ".join(parts)

df_music['raw_text'] = df_music.apply(combine_text_columns, axis=1)

print("\nContoh teks mentah quote :", df_http['quote_raw'].iloc[0])
print("Contoh teks mentah music :", df_music['raw_text'].iloc[0])


Kolom teks yang dipakai untuk preprocessing music: ['trackName', 'artistName']

Contoh teks mentah quote : “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
Contoh teks mentah music : Love Keyshia Cole


### **Case Folding**

In [ ]:
df_http['quote_step1_casefold'] = df_http['quote_raw'].str.lower()
df_music['music_step1_casefold'] = df_music['raw_text'].str.lower()

print("Contoh quote sebelum :", df_http['quote_raw'].iloc[0])
print("Contoh quote sesudah :", df_http['quote_step1_casefold'].iloc[0])
print()
print("Contoh music sebelum :", df_music['raw_text'].iloc[0])
print("Contoh music sesudah :", df_music['music_step1_casefold'].iloc[0])


Contoh quote sebelum : “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
Contoh quote sesudah : “the world as we have created it is a process of our thinking. it cannot be changed without changing our thinking.”

Contoh music sebelum : Love Keyshia Cole
Contoh music sesudah : love keyshia cole


### **Cleaning (Hapus Tanda Baca, Angka, dan Karakter Khusus)**

In [ ]:
def clean_text(text):
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_http['quote_step2_clean'] = df_http['quote_step1_casefold'].apply(clean_text)
df_music['music_step2_clean'] = df_music['music_step1_casefold'].apply(clean_text)

print("Contoh quote sebelum :", df_http['quote_step1_casefold'].iloc[0])
print("Contoh quote sesudah :", df_http['quote_step2_clean'].iloc[0])
print()
print("Contoh music sebelum :", df_music['music_step1_casefold'].iloc[0])
print("Contoh music sesudah :", df_music['music_step2_clean'].iloc[0])

Contoh quote sebelum : “the world as we have created it is a process of our thinking. it cannot be changed without changing our thinking.”
Contoh quote sesudah : “the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”

Contoh music sebelum : love keyshia cole
Contoh music sesudah : love keyshia cole


### **Tokenization**

In [ ]:
df_http['quote_step3_tokens'] = df_http['quote_step2_clean'].apply(word_tokenize)
df_music['music_step3_tokens'] = df_music['music_step2_clean'].apply(word_tokenize)

print("Contoh quote sebelum :", df_http['quote_step2_clean'].iloc[0])
print("Contoh quote sesudah :", df_http['quote_step3_tokens'].iloc[0])
print()
print("Contoh music sebelum :", df_music['music_step2_clean'].iloc[0])
print("Contoh music sesudah :", df_music['music_step3_tokens'].iloc[0])

Contoh quote sebelum : “the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
Contoh quote sesudah : ['“', 'the', 'world', 'as', 'we', 'have', 'created', 'it', 'is', 'a', 'process', 'of', 'our', 'thinking', 'it', 'can', 'not', 'be', 'changed', 'without', 'changing', 'our', 'thinking', '”']

Contoh music sebelum : love keyshia cole
Contoh music sesudah : ['love', 'keyshia', 'cole']


### **Stopword Removal**

In [ ]:
def remove_stopwords(tokens):
    return [t for t in tokens if t not in stop_words]

df_http['quote_step4_nostop'] = df_http['quote_step3_tokens'].apply(remove_stopwords)
df_music['music_step4_nostop'] = df_music['music_step3_tokens'].apply(remove_stopwords)

print("Contoh quote sebelum :", df_http['quote_step3_tokens'].iloc[0])
print("Contoh quote sesudah :", df_http['quote_step4_nostop'].iloc[0])
print()
print("Contoh music sebelum :", df_music['music_step3_tokens'].iloc[0])
print("Contoh music sesudah :", df_music['music_step4_nostop'].iloc[0])


Contoh quote sebelum : ['“', 'the', 'world', 'as', 'we', 'have', 'created', 'it', 'is', 'a', 'process', 'of', 'our', 'thinking', 'it', 'can', 'not', 'be', 'changed', 'without', 'changing', 'our', 'thinking', '”']
Contoh quote sesudah : ['“', 'world', 'created', 'process', 'thinking', 'changed', 'without', 'changing', 'thinking', '”']

Contoh music sebelum : ['love', 'keyshia', 'cole']
Contoh music sesudah : ['love', 'keyshia', 'cole']


### **Lemmatization**

In [ ]:
def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(t) for t in tokens]

df_http['quote_step5_lemma'] = df_http['quote_step4_nostop'].apply(lemmatize_tokens)
df_music['music_step5_lemma'] = df_music['music_step4_nostop'].apply(lemmatize_tokens)

df_http['quote_final'] = df_http['quote_step5_lemma'].apply(lambda toks: " ".join(toks))
df_music['music_final'] = df_music['music_step5_lemma'].apply(lambda toks: " ".join(toks))

print("Contoh quote sebelum :", df_http['quote_step4_nostop'].iloc[0])
print("Contoh quote sesudah :", df_http['quote_step5_lemma'].iloc[0])
print("Teks akhir quote     :", df_http['quote_final'].iloc[0])
print()
print("Contoh music sebelum :", df_music['music_step4_nostop'].iloc[0])
print("Contoh music sesudah :", df_music['music_step5_lemma'].iloc[0])
print("Teks akhir music     :", df_music['music_final'].iloc[0])


Contoh quote sebelum : ['“', 'world', 'created', 'process', 'thinking', 'changed', 'without', 'changing', 'thinking', '”']
Contoh quote sesudah : ['“', 'world', 'created', 'process', 'thinking', 'changed', 'without', 'changing', 'thinking', '”']
Teks akhir quote     : “ world created process thinking changed without changing thinking ”

Contoh music sebelum : ['love', 'keyshia', 'cole']
Contoh music sesudah : ['love', 'keyshia', 'cole']
Teks akhir music     : love keyshia cole


## **5. Rekap Hasil Preprocessing**

In [ ]:

print("=== Rekap HTTP Scraping (quotes.toscrape.com) ===")
display(df_http[['quote_raw', 'quote_step1_casefold', 'quote_step2_clean',
                  'quote_step3_tokens', 'quote_step4_nostop', 'quote_step5_lemma',
                  'quote_final']].head())

df_http.to_csv("hasil_preprocessing_http.csv", index=False)
print("File tersimpan: hasil_preprocessing_http.csv")

print("\n=== Rekap API Scraping (music-to-scrape.org) ===")
display(df_music[['raw_text', 'music_step1_casefold', 'music_step2_clean',
                   'music_step3_tokens', 'music_step4_nostop', 'music_step5_lemma',
                   'music_final']].head())

df_music.to_csv("hasil_preprocessing_music_api.csv", index=False)
print("File tersimpan: hasil_preprocessing_music_api.csv")

=== Rekap HTTP Scraping (quotes.toscrape.com) ===


,quote_raw,quote_step1_casefold,quote_step2_clean,quote_step3_tokens,quote_step4_nostop,quote_step5_lemma,quote_final
0,“The world as we have created it is a process ...,“the world as we have created it is a process ...,“the world as we have created it is a process ...,"[“, the, world, as, we, have, created, it, is,...","[“, world, created, process, thinking, changed...","[“, world, created, process, thinking, changed...",“ world created process thinking changed witho...
1,"“It is our choices, Harry, that show what we t...","“it is our choices, harry, that show what we t...",“it is our choices harry that show what we tru...,"[“, it, is, our, choices, harry, that, show, w...","[“, choices, harry, show, truly, far, abilitie...","[“, choice, harry, show, truly, far, ability, ”]",“ choice harry show truly far ability ”
2,“There are only two ways to live your life. On...,“there are only two ways to live your life. on...,“there are only two ways to live your life one...,"[“, there, are, only, two, ways, to, live, you...","[“, two, ways, live, life, one, though, nothin...","[“, two, way, live, life, one, though, nothing...",“ two way live life one though nothing miracle...
3,"“The person, be it gentleman or lady, who has ...","“the person, be it gentleman or lady, who has ...",“the person be it gentleman or lady who has no...,"[“, the, person, be, it, gentleman, or, lady, ...","[“, person, gentleman, lady, pleasure, good, n...","[“, person, gentleman, lady, pleasure, good, n...",“ person gentleman lady pleasure good novel mu...
4,"“Imperfection is beauty, madness is genius and...","“imperfection is beauty, madness is genius and...",“imperfection is beauty madness is genius and ...,"[“, imperfection, is, beauty, madness, is, gen...","[“, imperfection, beauty, madness, genius, bet...","[“, imperfection, beauty, madness, genius, bet...",“ imperfection beauty madness genius better ab...


File tersimpan: hasil_preprocessing_http.csv

=== Rekap API Scraping (music-to-scrape.org) ===


,raw_text,music_step1_casefold,music_step2_clean,music_step3_tokens,music_step4_nostop,music_step5_lemma,music_final
0,Love Keyshia Cole,love keyshia cole,love keyshia cole,"[love, keyshia, cole]","[love, keyshia, cole]","[love, keyshia, cole]",love keyshia cole
1,LOVE. (feat. Zacari) Kendrick Lamar,love. (feat. zacari) kendrick lamar,love feat zacari kendrick lamar,"[love, feat, zacari, kendrick, lamar]","[love, feat, zacari, kendrick, lamar]","[love, feat, zacari, kendrick, lamar]",love feat zacari kendrick lamar
2,Love Musiq Soulchild,love musiq soulchild,love musiq soulchild,"[love, musiq, soulchild]","[love, musiq, soulchild]","[love, musiq, soulchild]",love musiq soulchild
3,(dream) salvia palth,(dream) salvia palth,dream salvia palth,"[dream, salvia, palth]","[dream, salvia, palth]","[dream, salvia, palth]",dream salvia palth
4,"Dream Robxrt, the most romantic man in hanoi &...","dream robxrt, the most romantic man in hanoi &...",dream robxrt the most romantic man in hanoi hvggi,"[dream, robxrt, the, most, romantic, man, in, ...","[dream, robxrt, romantic, man, hanoi, hvggi]","[dream, robxrt, romantic, man, hanoi, hvggi]",dream robxrt romantic man hanoi hvggi


File tersimpan: hasil_preprocessing_music_api.csv


## **6. Visualisasi Word Cloud**



In [ ]:
# 1. Visualisasi Word Cloud untuk HTTP Scraping (Quotes)
text_quotes = ' '.join(df_http['quote_final'].dropna().astype(str))
wc_quotes = WordCloud(width=800, height=400, background_color='white', colormap='viridis', max_words=100).generate(text_quotes)

plt.figure(figsize=(10, 5))
plt.imshow(wc_quotes, interpolation='bilinear')
plt.axis('off')
plt.title("Word Cloud - Hasil Pre-processing Quotes (HTTP Scraping)", fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

# 2. Visualisasi Word Cloud untuk API Scraping (Music/Lagu)
text_music = ' '.join(df_music['music_final'].dropna().astype(str))
wc_music = WordCloud(width=800, height=400, background_color='white', colormap='plasma', max_words=100).generate(text_music)

plt.figure(figsize=(10, 5))
plt.imshow(wc_music, interpolation='bilinear')
plt.axis('off')
plt.title("Word Cloud - Hasil Pre-processing Music (API Scraping)", fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()